# Confirmatory Activation Probe (A100)

Evaluate the fixed net-torque probe on probe_test after fitting on probe_train and probe_dev.

[实验设置与运行说明](../../../docs/experiment_notes/03_section44_readout_correction/e7d5f31dfe5fac28.md)


In [ ]:
# Run this helper once. Every later code cell uses %%cell_progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 15 tracked cells.')


## 1. Setup

The first cells mount Drive, install the local toolkit, stage Dataset v1 on the Colab SSD, and read the previous discovery artifacts before probe_test is loaded.


In [ ]:
%%cell_progress 1/15 Mount Drive and locate project
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (path for path in PROJECT_CANDIDATES if (path / 'toolkit' / 'pyproject.toml').is_file()),
    None,
)
assert PROJECT_ROOT is not None, 'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/15 Install and verify environment
import importlib
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], check=False)
cell_stage('Installing pinned analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'tqdm>=4.66',
    'scikit-learn>=1.5', 'pandas>=2.2', 'matplotlib>=3.8',
])
cell_stage('Installing VisualTorque toolkit')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', str(TOOLKIT_ROOT)
])
SRC_ROOT = TOOLKIT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()

cell_stage('Checking multimodal imports')
import torch
import torchvision
import visual_torque_toolkit
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter
required_adapter_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(hasattr(HuggingFaceMultimodalAdapter, name) for name in required_adapter_features), (
    'Notebook/toolkit version mismatch. Sync toolkit/src to Drive and restart the runtime.'
)
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit:', visual_torque_toolkit.__file__)
print('Multimodal imports: OK')


In [ ]:
%%cell_progress 3/15 Verify A100 and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime before continuing.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login was skipped. Details:', exc)


In [ ]:
%%cell_progress 4/15 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all((path / name).is_file() for name in REQUIRED_DATASET_FILES)

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [member for member in bundle.infolist() if not member.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


In [ ]:
%%cell_progress 5/15 Lock discovery choice and create confirmatory output
cell_stage('Reading frozen discovery artifacts')
import hashlib
import json
from datetime import datetime, timezone

# Run 1 (default).
MODEL_ID = 'Qwen/Qwen3.5-9B'
BATCH_SIZE = 4

# Run 2: uncomment both lines below in a fresh runtime.
# MODEL_ID = 'google/gemma-4-12B-it'
# BATCH_SIZE = 2

FROZEN_EXPECTATIONS = {
    'Qwen/Qwen3.5-9B': {'layer_index': 13, 'alpha': 0.01},
    'google/gemma-4-12B-it': {'layer_index': 7, 'alpha': 100.0},
}
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
LAYER_PATH_OVERRIDE = None
SHARD_EXAMPLES = 96
N_SHUFFLES = 200
BOOTSTRAP_REPLICATES = 5000
BALANCE_THRESHOLD = 0.5
SEED = 20260818

def model_slug(model_id):
    return model_id.replace('/', '__')

MODEL_RESULT_ROOT = PROJECT_ROOT / 'results' / model_slug(MODEL_ID)
DISCOVERY_CAPTURE_DIR = MODEL_RESULT_ROOT / 'activation_discovery_v1'
DISCOVERY_PROBE_DIR = DISCOVERY_CAPTURE_DIR / 'probe_results'
DISCOVERY_SELECTION_PATH = DISCOVERY_PROBE_DIR / 'discovery_selection.json'
DISCOVERY_CONFIG_PATH = DISCOVERY_CAPTURE_DIR / 'capture_config.json'
DISCOVERY_SCENE_MEAN_PATH = DISCOVERY_CAPTURE_DIR / 'scene_mean_activations.pt'
for required in (
    DISCOVERY_SELECTION_PATH, DISCOVERY_CONFIG_PATH, DISCOVERY_SCENE_MEAN_PATH,
):
    assert required.is_file(), f'Missing discovery artifact: {required}'

discovery_selection = json.loads(DISCOVERY_SELECTION_PATH.read_text(encoding='utf-8'))
discovery_config = json.loads(DISCOVERY_CONFIG_PATH.read_text(encoding='utf-8'))
discovery_identity = discovery_config['identity']
assert discovery_selection['model_id'] == MODEL_ID
assert discovery_identity['model_id'] == MODEL_ID
assert discovery_selection['probe_test_used'] is False
assert discovery_selection['selection_data'] == ['probe_train', 'probe_dev']
frozen_choice = discovery_selection['candidate_net_torque_layer']
SELECTED_LAYER_INDEX = int(frozen_choice['layer_index'])
FROZEN_ALPHA = float(frozen_choice['selected_alpha'])
expected = FROZEN_EXPECTATIONS[MODEL_ID]
assert SELECTED_LAYER_INDEX == expected['layer_index'], (
    SELECTED_LAYER_INDEX, expected['layer_index']
)
assert abs(FROZEN_ALPHA - expected['alpha']) < 1e-12, (
    FROZEN_ALPHA, expected['alpha']
)
MODEL_REVISION = str(discovery_identity['resolved_revision'])

TEST_CAPTURE_DIR = MODEL_RESULT_ROOT / 'activation_confirmatory_v1'
RESULT_DIR = TEST_CAPTURE_DIR / 'confirmatory_results'
TEST_CAPTURE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# This lock is written before the next cell reads probe_test. Its identity
# deliberately excludes timestamps so rerunning this cell is idempotent.
PROTOCOL_LOCK_PATH = TEST_CAPTURE_DIR / 'protocol_lock.json'
protocol_identity = {
    'schema_version': 'activation-confirmatory-protocol-1.0',
    'model_id': MODEL_ID,
    'resolved_revision': MODEL_REVISION,
    'discovery_selection_path': str(DISCOVERY_SELECTION_PATH),
    'selected_layer_index': SELECTED_LAYER_INDEX,
    'frozen_ridge_alpha': FROZEN_ALPHA,
    'primary_target': 'net_torque',
    'fit_splits': ['probe_train', 'probe_dev'],
    'evaluation_split': 'probe_test',
    'representation': 'mean_of_six_permutation_final_prompt_states',
    'balance_threshold': BALANCE_THRESHOLD,
    'shuffle_replicates': N_SHUFFLES,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'confirmation_rule': (
        'test R2 > 0; stratified-bootstrap R2 lower 95% bound > 0; '
        'one-sided shuffled-label p <= 0.05'
    ),
    'seed': SEED,
}
if PROTOCOL_LOCK_PATH.is_file():
    previous_lock = json.loads(
        PROTOCOL_LOCK_PATH.read_text(encoding='utf-8')
    )
    assert previous_lock.get('identity') == protocol_identity, (
        'Existing protocol lock differs. Use a new versioned output directory.'
    )
else:
    temporary_lock = PROTOCOL_LOCK_PATH.with_suffix('.json.tmp')
    temporary_lock.write_text(
        json.dumps({
            'identity': protocol_identity,
            'locked_before_probe_test_at_utc': (
                datetime.now(timezone.utc).isoformat()
            ),
        }, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary_lock.replace(PROTOCOL_LOCK_PATH)
print('Model:', MODEL_ID)
print('Pinned revision:', MODEL_REVISION)
print('Frozen layer:', SELECTED_LAYER_INDEX)
print('Frozen ridge alpha:', FROZEN_ALPHA)
print('Confirmatory output:', TEST_CAPTURE_DIR)


In [ ]:
%%cell_progress 6/15 Load the untouched probe_test examples
cell_stage('Reading Dataset v1 scenes')
from collections import Counter

import pandas as pd
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

all_scenes = read_jsonl(DATASET_DIR / 'scenes.jsonl')
scene_by_id = {str(scene['scene_id']): scene for scene in all_scenes}
assert len(scene_by_id) == len(all_scenes)

cell_stage('Reconstructing all image-rule outcome prompts in probe_test')
test_example_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=Path('/content/unused_activation_confirmatory_output'),
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    splits=('probe_test',),
    batch_size=BATCH_SIZE,
    generation_every=0,
    seed=SEED,
)
test_examples = tuple(load_behavior_examples(test_example_config))
assert test_examples, 'probe_test returned no outcome examples.'
assert {example.split for example in test_examples} == {'probe_test'}
test_scene_ids = sorted({str(example.scene_id) for example in test_examples})
counts = Counter(str(example.scene_id) for example in test_examples)
assert set(counts.values()) == {6}, Counter(counts.values())
assert all(scene_by_id[scene_id]['split'] == 'probe_test' for scene_id in test_scene_ids)

def physical_targets(scene):
    left = [stack for stack in scene['stacks'] if stack['side'] == 'left']
    right = [stack for stack in scene['stacks'] if stack['side'] == 'right']
    derived = scene['derived']
    return {
        'left_units': int(sum(int(stack['units']) for stack in left)),
        'right_units': int(sum(int(stack['units']) for stack in right)),
        'left_distance': int(left[0]['distance']) if len(left) == 1 else None,
        'right_distance': int(right[0]['distance']) if len(right) == 1 else None,
        'left_torque': int(derived['left_torque']),
        'right_torque': int(derived['right_torque']),
        'net_torque': int(derived['net_torque']),
        'outcome': str(derived['outcome']),
    }

targets_by_scene = {
    scene_id: physical_targets(scene_by_id[scene_id]) for scene_id in test_scene_ids
}
test_manifest_rows = [
    {
        'schema_version': 'activation-confirmatory-manifest-1.0',
        'scene_id': scene_id,
        'split': 'probe_test',
        'problem_type': str(scene_by_id[scene_id]['problem_type']),
        **targets_by_scene[scene_id],
    }
    for scene_id in test_scene_ids
]
test_manifest_path = TEST_CAPTURE_DIR / 'probe_test_manifest.jsonl'
temporary_manifest = test_manifest_path.with_suffix('.jsonl.tmp')
with temporary_manifest.open('w', encoding='utf-8') as handle:
    for record in test_manifest_rows:
        handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + '\n')
temporary_manifest.replace(test_manifest_path)
print('probe_test scenes:', len(test_scene_ids))
print('probe_test permutation examples:', len(test_examples))
print('Problem types:', Counter(scene_by_id[s]['problem_type'] for s in test_scene_ids))


In [ ]:
%%cell_progress 7/15 Load the pinned model and selected decoder layer
cell_stage('Loading processor and pinned model weights')
adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
    device_map=DEVICE_MAP,
)
display(adapter.metadata)

cell_stage('Resolving language-decoder blocks')
def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model, override=None):
    candidates = [override] if override else [
        discovery_identity.get('layer_path'),
        'model.language_model.layers',
        'model.text_model.layers',
        'language_model.model.layers',
        'language_model.layers',
        'model.model.layers',
        'model.layers',
    ]
    failures = []
    for path in candidates:
        if not path:
            continue
        try:
            layers = resolve_module_path(model, path)
        except (AttributeError, KeyError) as exc:
            failures.append(f'{path}: {exc}')
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
        failures.append(
            f'{path}: not a decoder ModuleList (type={type(layers).__name__})'
        )
    raise RuntimeError('Could not locate decoder layers:\n' + '\n'.join(failures))

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model, LAYER_PATH_OVERRIDE)
assert len(DECODER_LAYERS) == int(discovery_identity['decoder_layer_count'])
assert 0 <= SELECTED_LAYER_INDEX < len(DECODER_LAYERS)
resolved_now = adapter.metadata.get('resolved_revision')
if resolved_now is not None:
    assert str(resolved_now) == MODEL_REVISION, (resolved_now, MODEL_REVISION)
SELECTED_NORMALIZED_DEPTH = (SELECTED_LAYER_INDEX + 1) / len(DECODER_LAYERS)
SELECTED_LAYER = DECODER_LAYERS[SELECTED_LAYER_INDEX]
print('Model class:', type(adapter.model).__name__)
print('Layer path:', LAYER_PATH)
print('Decoder layers:', len(DECODER_LAYERS))
print('Locked layer/depth:', SELECTED_LAYER_INDEX, round(SELECTED_NORMALIZED_DEPTH, 4))


## 2. Resumable probe_test activation capture

Only the frozen candidate layer is captured.


In [ ]:
%%cell_progress 8/15 Define safe single-layer activation capture
cell_stage('Defining hashing, hooks, records, and resumable shards')
import os

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        tensor = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        tensor = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        tensor = output.last_hidden_state
    else:
        raise TypeError(
            f'Unsupported decoder-layer output type: {type(output).__name__}'
        )
    if tensor.ndim != 3:
        raise ValueError(f'Expected [batch, sequence, hidden], got {tuple(tensor.shape)}')
    return tensor

def capture_activation_batch(examples):
    if not examples:
        raise ValueError('capture batch must not be empty')
    if any(example.labels != examples[0].labels for example in examples):
        raise ValueError('all examples in a batch must use the same labels')
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError(
            'Confirmatory capture requires single-token A/B/C scoring for this model.'
        )
    fixed_prefix, token_by_label = plan
    labels = list(examples[0].labels)
    conversations = [example.messages() for example in examples]
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(conversations, add_generation_prompt=True)),
        fixed_prefix,
    )
    captured = {}

    def hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        captured['activation'] = hidden[:, -1, :].detach().to(
            device='cpu', dtype=torch.float16
        )

    handle = SELECTED_LAYER.register_forward_hook(hook)
    started = time.perf_counter()
    try:
        with torch.inference_mode():
            output = adapter.model(**inputs, use_cache=False)
            next_logprobs = torch.log_softmax(
                output.logits[:, -1, :].float(), dim=-1
            )
    finally:
        handle.remove()
    if 'activation' not in captured:
        raise RuntimeError('Selected decoder hook did not fire.')
    activations = captured['activation']
    if activations.shape[0] != len(examples):
        raise RuntimeError(f'Activation batch mismatch: {tuple(activations.shape)}')
    score_rows = torch.stack([
        next_logprobs[:, token_by_label[label]].detach().cpu()
        for label in labels
    ], dim=1)
    latency = (time.perf_counter() - started) / len(examples)
    records = []
    for row, example in enumerate(examples):
        scores = {
            label: float(score_rows[row, column])
            for column, label in enumerate(labels)
        }
        predicted_label = max(labels, key=lambda label: scores[label])
        targets = targets_by_scene[str(example.scene_id)]
        records.append({
            'schema_version': 'activation-confirmatory-example-1.0',
            'example_id': example.example_id,
            'question_id': example.question_id,
            'scene_id': str(example.scene_id),
            'split': example.split,
            'problem_type': example.problem_type,
            'variant': example.variant,
            'gold_semantic': str(example.gold_semantic),
            'gold_label': example.gold_label,
            'label_to_semantic': example.label_to_semantic,
            'predicted_label': predicted_label,
            'predicted_semantic': str(example.label_to_semantic[predicted_label]),
            'candidate_logprobs': scores,
            'latency_seconds': latency,
            **targets,
        })
    return {'activations': activations, 'records': records, 'labels': labels}

def torch_load_trusted(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def json_fallback(value):
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, 'item'):
        return value.item()
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(
            value, indent=2, ensure_ascii=False, default=json_fallback
        ) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

def capture_identity():
    return {
        'schema_version': 'activation-confirmatory-capture-1.0',
        'model_id': MODEL_ID,
        'requested_revision': MODEL_REVISION,
        'resolved_revision': adapter.metadata.get('resolved_revision'),
        'dtype': MODEL_DTYPE,
        'dataset_fingerprint': test_example_config.dataset_fingerprint(),
        'dataset_scenes_sha256': sha256_file(DATASET_DIR / 'scenes.jsonl'),
        'dataset_splits_sha256': sha256_file(DATASET_DIR / 'splits.json'),
        'discovery_selection_sha256': sha256_file(DISCOVERY_SELECTION_PATH),
        'protocol_lock_sha256': sha256_file(PROTOCOL_LOCK_PATH),
        'example_count': len(test_examples),
        'scene_count': len(test_scene_ids),
        'allowed_splits': ['probe_test'],
        'condition': 'image_rule',
        'question_kind': 'outcome',
        'permutations_per_scene': 6,
        'layer_path': LAYER_PATH,
        'decoder_layer_count': len(DECODER_LAYERS),
        'selected_layer_index': SELECTED_LAYER_INDEX,
        'selected_normalized_depth': SELECTED_NORMALIZED_DEPTH,
        'frozen_ridge_alpha': FROZEN_ALPHA,
        'capture_position': 'final_token_after_generation_prompt_and_fixed_prefix',
        'activation_dtype': 'float16',
        'seed': SEED,
    }

def run_resumable_capture():
    identity = capture_identity()
    config_path = TEST_CAPTURE_DIR / 'capture_config.json'
    if config_path.is_file():
        previous = json.loads(config_path.read_text(encoding='utf-8'))
        if previous.get('identity') != identity:
            raise ValueError(
                'Existing confirmatory directory belongs to a different experiment. '
                'Use a new versioned output directory rather than mixing test activations.'
            )
    else:
        write_json_atomic(config_path, {
            'identity': identity,
            'adapter_metadata': adapter.metadata,
            'probe_test_opened_at_utc': datetime.now(timezone.utc).isoformat(),
            'protocol_note': (
                'Layer and ridge alpha were frozen on probe_train/probe_dev before '
                'probe_test capture.'
            ),
        })

    shard_paths = sorted(TEST_CAPTURE_DIR.glob('shard_*.pt'))
    completed = set()
    for path in tqdm(shard_paths, desc='Reading completed shards', unit='shard'):
        payload = torch_load_trusted(path)
        if int(payload['selected_layer_index']) != SELECTED_LAYER_INDEX:
            raise RuntimeError(f'Layer mismatch in {path.name}')
        for record in payload['records']:
            example_id = str(record['example_id'])
            if example_id in completed:
                raise RuntimeError(f'Duplicate captured example: {example_id}')
            completed.add(example_id)
    expected_ids = {example.example_id for example in test_examples}
    if not completed <= expected_ids:
        raise RuntimeError('Existing shards contain examples outside probe_test.')
    remaining = [
        example for example in test_examples if example.example_id not in completed
    ]
    print('Already captured:', len(completed), '| remaining:', len(remaining))

    next_shard = 0
    if shard_paths:
        next_shard = max(
            int(path.stem.split('_')[-1]) for path in shard_paths
        ) + 1
    pending_activations = []
    pending_records = []

    def flush():
        nonlocal next_shard, pending_activations, pending_records
        if not pending_records:
            return
        payload = {
            'schema_version': 'activation-confirmatory-shard-1.0',
            'selected_layer_index': SELECTED_LAYER_INDEX,
            'selected_normalized_depth': SELECTED_NORMALIZED_DEPTH,
            'activations': torch.cat(pending_activations, dim=0),
            'records': pending_records,
        }
        destination = TEST_CAPTURE_DIR / f'shard_{next_shard:05d}.pt'
        temporary = TEST_CAPTURE_DIR / f'shard_{next_shard:05d}.pt.tmp'
        torch.save(payload, temporary)
        temporary.replace(destination)
        next_shard += 1
        pending_activations = []
        pending_records = []

    progress = tqdm(
        total=len(remaining), desc='probe_test activation capture',
        unit='example', dynamic_ncols=True,
    )
    for start in range(0, len(remaining), BATCH_SIZE):
        batch = remaining[start:start + BATCH_SIZE]
        captured_batch = capture_activation_batch(batch)
        pending_activations.append(captured_batch['activations'])
        pending_records.extend(captured_batch['records'])
        progress.update(len(batch))
        progress.set_postfix_str(
            f"shard buffer {len(pending_records)}/{SHARD_EXAMPLES}",
            refresh=True,
        )
        if len(pending_records) >= SHARD_EXAMPLES:
            flush()
    flush()
    progress.close()

    index_rows = []
    seen = set()
    final_paths = sorted(TEST_CAPTURE_DIR.glob('shard_*.pt'))
    for path in tqdm(final_paths, desc='Building test capture index', unit='shard'):
        payload = torch_load_trusted(path)
        for row, record in enumerate(payload['records']):
            if record['example_id'] in seen:
                raise RuntimeError(f"Duplicate example: {record['example_id']}")
            seen.add(record['example_id'])
            index_rows.append({'shard': path.name, 'row': row, **record})
    if seen != expected_ids:
        raise RuntimeError(
            f'Capture incomplete: expected {len(expected_ids)}, found {len(seen)}'
        )
    index_path = TEST_CAPTURE_DIR / 'capture_index.jsonl'
    temporary_index = index_path.with_suffix('.jsonl.tmp')
    with temporary_index.open('w', encoding='utf-8') as handle:
        for record in index_rows:
            handle.write(
                json.dumps(record, ensure_ascii=False, sort_keys=True) + '\n'
            )
    temporary_index.replace(index_path)
    summary = {
        'expected_examples': len(expected_ids),
        'completed_examples': len(seen),
        'scene_count': len({record['scene_id'] for record in index_rows}),
        'shard_count': len(final_paths),
        'index_path': str(index_path),
    }
    write_json_atomic(TEST_CAPTURE_DIR / 'capture_summary.json', summary)
    return summary

print('Confirmatory capture utilities ready.')


In [ ]:
%%cell_progress 9/15 Run mandatory confirmatory smoke test
cell_stage('Capturing two probe_test examples at the frozen layer')
smoke_examples = test_examples[:2]
smoke_capture = capture_activation_batch(smoke_examples)
smoke_activations = smoke_capture['activations']
assert smoke_activations.shape[0] == 2
assert smoke_activations.ndim == 2
assert torch.isfinite(smoke_activations).all()
assert float(smoke_activations.float().std()) > 0

cell_stage('Checking hook scoring against the behavior adapter')
reference = adapter.score_batch(smoke_examples)
for captured_record, reference_record in zip(smoke_capture['records'], reference):
    for label in ('A', 'B', 'C'):
        difference = abs(
            captured_record['candidate_logprobs'][label]
            - reference_record.logprob_by_label[label]
        )
        assert difference < 1e-3, (label, difference)
print('Smoke activation shape:', tuple(smoke_activations.shape))
print('Scoring parity: passed')


In [ ]:
%%cell_progress 10/15 Capture or resume the full probe_test split
cell_stage('Capturing frozen-layer probe_test activations')
capture_summary = run_resumable_capture()
display(capture_summary)


In [ ]:
%%cell_progress 11/15 Release GPU and assemble train/dev plus test scene means
cell_stage('Releasing model weights before CPU analysis')
import gc
del smoke_capture, smoke_activations, reference
del adapter, SELECTED_LAYER, DECODER_LAYERS
gc.collect()
torch.cuda.empty_cache()
print('GPU cache released.')

cell_stage('Loading probe_test activation shards')
import numpy as np
activation_parts = []
record_rows = []
for path in tqdm(
    sorted(TEST_CAPTURE_DIR.glob('shard_*.pt')),
    desc='Loading test shards', unit='shard',
):
    payload = torch_load_trusted(path)
    activation_parts.append(payload['activations'].float().numpy())
    record_rows.extend(payload['records'])
test_variant_activations = np.concatenate(activation_parts, axis=0)
test_variant_metadata = pd.DataFrame(record_rows)
assert len(test_variant_metadata) == len(test_examples)
assert test_variant_metadata['example_id'].is_unique
assert set(test_variant_metadata['split']) == {'probe_test'}
assert test_variant_activations.shape[0] == len(test_variant_metadata)

cell_stage('Averaging six test permutations per scene')
test_scene_activation_rows = []
test_scene_metadata_rows = []
variant_scene_ids = test_variant_metadata['scene_id'].astype(str).to_numpy()
for scene_id in tqdm(test_scene_ids, desc='Test permutation averaging', unit='scene'):
    indices = np.flatnonzero(variant_scene_ids == scene_id)
    group = test_variant_metadata.iloc[indices]
    assert len(indices) == 6
    assert group['variant'].nunique() == 6
    for column in (
        'split', 'problem_type', 'gold_semantic', 'left_units', 'right_units',
        'left_distance', 'right_distance', 'left_torque', 'right_torque',
        'net_torque', 'outcome',
    ):
        assert group[column].nunique(dropna=False) == 1, (scene_id, column)
    test_scene_activation_rows.append(
        test_variant_activations[indices].mean(axis=0)
    )
    first = group.iloc[0]
    test_scene_metadata_rows.append({
        column: first[column] for column in (
            'scene_id', 'split', 'problem_type', 'gold_semantic',
            'left_units', 'right_units', 'left_distance', 'right_distance',
            'left_torque', 'right_torque', 'net_torque', 'outcome',
        )
    })
test_scene_activations = np.stack(test_scene_activation_rows).astype(np.float32)
test_scene_metadata = pd.DataFrame(test_scene_metadata_rows)
assert len(test_scene_metadata) == len(test_scene_ids)

test_scene_mean_path = TEST_CAPTURE_DIR / 'probe_test_scene_mean_activations.pt'
torch.save({
    'schema_version': 'activation-confirmatory-scene-mean-1.0',
    'selected_layer_index': SELECTED_LAYER_INDEX,
    'selected_normalized_depth': SELECTED_NORMALIZED_DEPTH,
    'scene_ids': test_scene_ids,
    'activations': torch.from_numpy(test_scene_activations).to(torch.float16),
    'metadata': test_scene_metadata.to_dict(orient='records'),
}, test_scene_mean_path)

cell_stage('Loading frozen discovery train/dev scene means')
discovery_payload = torch_load_trusted(DISCOVERY_SCENE_MEAN_PATH)
discovery_layer_indices = [
    int(value) for value in discovery_payload['layer_indices']
]
assert SELECTED_LAYER_INDEX in discovery_layer_indices
discovery_layer_position = discovery_layer_indices.index(SELECTED_LAYER_INDEX)
discovery_scene_activations = (
    discovery_payload['activations'][:, discovery_layer_position, :]
    .float().numpy()
)
discovery_scene_metadata = pd.DataFrame(discovery_payload['metadata'])
assert set(discovery_scene_metadata['split']) <= {'probe_train', 'probe_dev'}
assert 'probe_test' not in set(discovery_scene_metadata['split'])
assert set(discovery_scene_metadata['scene_id'].astype(str)) & set(test_scene_ids) == set()

def scene_feature_row(scene):
    left = [stack for stack in scene['stacks'] if stack['side'] == 'left']
    right = [stack for stack in scene['stacks'] if stack['side'] == 'right']
    derived = scene['derived']
    return {
        'left_units': float(sum(stack['units'] for stack in left)),
        'right_units': float(sum(stack['units'] for stack in right)),
        'left_distance_sum': float(sum(stack['distance'] for stack in left)),
        'right_distance_sum': float(sum(stack['distance'] for stack in right)),
        'left_stack_count': float(len(left)),
        'right_stack_count': float(len(right)),
        'left_torque': float(derived['left_torque']),
        'right_torque': float(derived['right_torque']),
        'total_weight_difference': float(
            sum(stack['units'] for stack in left)
            - sum(stack['units'] for stack in right)
        ),
    }

fit_feature_frame = pd.DataFrame([
    scene_feature_row(scene_by_id[str(scene_id)])
    for scene_id in discovery_scene_metadata['scene_id']
])
test_feature_frame = pd.DataFrame([
    scene_feature_row(scene_by_id[str(scene_id)])
    for scene_id in test_scene_metadata['scene_id']
])
print('Fit scene activations:', discovery_scene_activations.shape)
print('Test scene activations:', test_scene_activations.shape)
print('Test scene means:', test_scene_mean_path)


## 3. Confirmatory analysis

The next cells refit one frozen net-torque probe on probe_train plus probe_dev and report probe_test metrics.


In [ ]:
%%cell_progress 12/15 Fit the frozen net-torque probe and shuffled nulls
cell_stage('Importing CPU probe utilities')
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, mean_absolute_error, r2_score,
)
from sklearn.preprocessing import StandardScaler

def safe_pearson(left, right):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    if len(left) < 2 or not np.std(left) or not np.std(right):
        return None
    return float(np.corrcoef(left, right)[0, 1])

def safe_spearman(left, right):
    left_rank = pd.Series(np.asarray(left, dtype=float)).rank(method='average')
    right_rank = pd.Series(np.asarray(right, dtype=float)).rank(method='average')
    return safe_pearson(left_rank.to_numpy(), right_rank.to_numpy())

def torque_to_outcome(values, threshold=BALANCE_THRESHOLD):
    values = np.asarray(values, dtype=float)
    return np.where(
        values > threshold, 'left',
        np.where(values < -threshold, 'right', 'balanced'),
    )

y_fit = pd.to_numeric(
    discovery_scene_metadata['net_torque'], errors='raise'
).to_numpy(dtype=float)
y_test = pd.to_numeric(
    test_scene_metadata['net_torque'], errors='raise'
).to_numpy(dtype=float)
gold_outcomes = test_scene_metadata['outcome'].astype(str).to_numpy()

activation_scaler = StandardScaler()
X_fit = activation_scaler.fit_transform(discovery_scene_activations)
X_test = activation_scaler.transform(test_scene_activations)
frozen_probe = Ridge(alpha=FROZEN_ALPHA, solver='lsqr', max_iter=5000)
frozen_probe.fit(X_fit, y_fit)
probe_prediction = frozen_probe.predict(X_test)
probe_outcome = torque_to_outcome(probe_prediction)

primary_point_metrics = {
    'test_r2': float(r2_score(y_test, probe_prediction)),
    'test_mae': float(mean_absolute_error(y_test, probe_prediction)),
    'test_pearson': safe_pearson(y_test, probe_prediction),
    'test_spearman': safe_spearman(y_test, probe_prediction),
    'test_torque_sign_accuracy': float(accuracy_score(gold_outcomes, probe_outcome)),
    'test_outcome_balanced_accuracy': float(
        balanced_accuracy_score(gold_outcomes, probe_outcome)
    ),
}
display(pd.DataFrame([primary_point_metrics]).round(4))

cell_stage('Running fixed shuffled-label null probes')
null_rows = []
shuffle_rng = np.random.default_rng(SEED + 7000)
for replicate in tqdm(
    range(N_SHUFFLES), desc='Shuffled-label null probes', unit='shuffle'
):
    shuffled_y = shuffle_rng.permutation(y_fit)
    null_probe = Ridge(alpha=FROZEN_ALPHA, solver='lsqr', max_iter=5000)
    null_probe.fit(X_fit, shuffled_y)
    null_prediction = null_probe.predict(X_test)
    null_rows.append({
        'replicate': replicate,
        'test_r2': float(r2_score(y_test, null_prediction)),
        'test_mae': float(mean_absolute_error(y_test, null_prediction)),
        'test_pearson': safe_pearson(y_test, null_prediction),
    })
shuffle_null = pd.DataFrame(null_rows)
shuffle_p_value = float(
    (1 + (shuffle_null['test_r2'] >= primary_point_metrics['test_r2']).sum())
    / (N_SHUFFLES + 1)
)
shuffle_null.to_csv(RESULT_DIR / 'shuffle_null_r2.csv', index=False)
print('Observed test R²:', round(primary_point_metrics['test_r2'], 4))
print('Null R² 95th percentile:', round(
    float(shuffle_null['test_r2'].quantile(0.95)), 4
))
print('One-sided empirical p:', shuffle_p_value)


In [ ]:
%%cell_progress 13/15 Fit preregistered shortcut and oracle-feature baselines
cell_stage('Fitting low-dimensional baselines on train plus dev only')
BASELINE_FEATURES = {
    'total_weight_difference': ['total_weight_difference'],
    'units_only': ['left_units', 'right_units'],
    'geometry_only': [
        'left_distance_sum', 'right_distance_sum',
        'left_stack_count', 'right_stack_count',
    ],
    'additive_primitives_no_products': [
        'left_units', 'right_units',
        'left_distance_sum', 'right_distance_sum',
        'left_stack_count', 'right_stack_count',
    ],
    'oracle_side_torques': ['left_torque', 'right_torque'],
}

def regression_metric_row(name, truth, prediction):
    predicted_outcome = torque_to_outcome(prediction)
    return {
        'model_id': MODEL_ID,
        'baseline': name,
        'fit_count': len(y_fit),
        'test_count': len(truth),
        'test_r2': float(r2_score(truth, prediction)),
        'test_mae': float(mean_absolute_error(truth, prediction)),
        'test_pearson': safe_pearson(truth, prediction),
        'test_spearman': safe_spearman(truth, prediction),
        'test_torque_sign_accuracy': float(
            accuracy_score(gold_outcomes, predicted_outcome)
        ),
        'test_outcome_balanced_accuracy': float(
            balanced_accuracy_score(gold_outcomes, predicted_outcome)
        ),
    }

baseline_rows = []
mean_prediction = np.full_like(y_test, y_fit.mean(), dtype=float)
baseline_rows.append(regression_metric_row('train_mean', y_test, mean_prediction))
baseline_predictions = {'train_mean': mean_prediction}
for name, columns in tqdm(
    BASELINE_FEATURES.items(), desc='Shortcut baselines', unit='baseline'
):
    estimator = LinearRegression()
    estimator.fit(
        fit_feature_frame[columns].to_numpy(dtype=float), y_fit
    )
    prediction = estimator.predict(
        test_feature_frame[columns].to_numpy(dtype=float)
    )
    baseline_predictions[name] = prediction
    baseline_rows.append(regression_metric_row(name, y_test, prediction))
baseline_predictions['activation_probe'] = probe_prediction
baseline_rows.append(regression_metric_row(
    'activation_probe', y_test, probe_prediction
))
shortcut_baselines = pd.DataFrame(baseline_rows).sort_values(
    'test_r2', ascending=False
)
shortcut_baselines.to_csv(RESULT_DIR / 'shortcut_baselines.csv', index=False)
display(shortcut_baselines.round(4))


In [ ]:
%%cell_progress 14/15 Bootstrap intervals and connect representation to behavior
cell_stage('Running stratified scene bootstrap')
def stratified_bootstrap_indices(labels, rng):
    labels = np.asarray(labels)
    parts = []
    for label in sorted(set(labels.tolist())):
        members = np.flatnonzero(labels == label)
        parts.append(rng.choice(members, size=len(members), replace=True))
    return np.concatenate(parts)

metric_functions = {
    'test_r2': lambda truth, pred, outcome_truth, outcome_pred: r2_score(truth, pred),
    'test_mae': lambda truth, pred, outcome_truth, outcome_pred: mean_absolute_error(
        truth, pred
    ),
    'test_pearson': lambda truth, pred, outcome_truth, outcome_pred: safe_pearson(
        truth, pred
    ),
    'test_spearman': lambda truth, pred, outcome_truth, outcome_pred: safe_spearman(
        truth, pred
    ),
    'test_torque_sign_accuracy': (
        lambda truth, pred, outcome_truth, outcome_pred:
        accuracy_score(outcome_truth, outcome_pred)
    ),
    'test_outcome_balanced_accuracy': (
        lambda truth, pred, outcome_truth, outcome_pred:
        balanced_accuracy_score(outcome_truth, outcome_pred)
    ),
}
bootstrap_values = {name: [] for name in metric_functions}
bootstrap_rng = np.random.default_rng(SEED + 9000)
for _ in tqdm(
    range(BOOTSTRAP_REPLICATES), desc='Scene bootstrap', unit='replicate'
):
    indices = stratified_bootstrap_indices(gold_outcomes, bootstrap_rng)
    truth = y_test[indices]
    prediction = probe_prediction[indices]
    outcome_truth = gold_outcomes[indices]
    outcome_prediction = probe_outcome[indices]
    for name, function in metric_functions.items():
        value = function(
            truth, prediction, outcome_truth, outcome_prediction
        )
        if value is not None and np.isfinite(value):
            bootstrap_values[name].append(float(value))

bootstrap_rows = []
for name, values in bootstrap_values.items():
    if not values:
        lower = upper = None
    else:
        lower, upper = np.quantile(values, [0.025, 0.975]).tolist()
    bootstrap_rows.append({
        'metric': name,
        'point': primary_point_metrics[name],
        'ci95_low': lower,
        'ci95_high': upper,
        'valid_replicates': len(values),
    })
bootstrap_intervals = pd.DataFrame(bootstrap_rows)
bootstrap_intervals.to_csv(
    RESULT_DIR / 'bootstrap_intervals.csv', index=False
)
display(bootstrap_intervals.round(4))

cell_stage('Aggregating the model answer over six permutations')
model_rows = []
for scene_id in test_scene_ids:
    group = test_variant_metadata[
        test_variant_metadata['scene_id'].astype(str).eq(scene_id)
    ]
    semantic_scores = {'left': [], 'right': [], 'balanced': []}
    for _, row in group.iterrows():
        for label in ('A', 'B', 'C'):
            semantic = str(row['label_to_semantic'][label])
            semantic_scores[semantic].append(
                float(row['candidate_logprobs'][label])
            )
    means = {
        semantic: float(np.mean(scores))
        for semantic, scores in semantic_scores.items()
    }
    prediction = max(means, key=means.get)
    gold = str(group.iloc[0]['outcome'])
    nongold = max(value for key, value in means.items() if key != gold)
    model_rows.append({
        'scene_id': scene_id,
        'model_predicted_outcome': prediction,
        'model_gold_margin': means[gold] - nongold,
        'model_score_left': means['left'],
        'model_score_right': means['right'],
        'model_score_balanced': means['balanced'],
    })
model_scene_scores = pd.DataFrame(model_rows)

prediction_frame = test_scene_metadata.copy()
prediction_frame['probe_predicted_net_torque'] = probe_prediction
prediction_frame['probe_predicted_outcome'] = probe_outcome
prediction_frame = prediction_frame.merge(
    model_scene_scores, on='scene_id', how='left', validate='one_to_one'
)
weight_difference = test_feature_frame['total_weight_difference'].to_numpy()
prediction_frame['weight_heuristic_outcome'] = np.where(
    weight_difference > 0, 'left',
    np.where(weight_difference < 0, 'right', 'balanced'),
)
prediction_frame['probe_correct'] = (
    prediction_frame['probe_predicted_outcome'] == prediction_frame['outcome']
)
prediction_frame['model_correct'] = (
    prediction_frame['model_predicted_outcome'] == prediction_frame['outcome']
)
prediction_frame['weight_heuristic_correct'] = (
    prediction_frame['weight_heuristic_outcome'] == prediction_frame['outcome']
)
prediction_frame['model_follows_weight_heuristic'] = (
    prediction_frame['model_predicted_outcome']
    == prediction_frame['weight_heuristic_outcome']
)
prediction_frame['representation_behavior_case'] = np.select(
    [
        prediction_frame['probe_correct'] & prediction_frame['model_correct'],
        prediction_frame['probe_correct'] & ~prediction_frame['model_correct'],
        ~prediction_frame['probe_correct'] & prediction_frame['model_correct'],
    ],
    ['both_correct', 'probe_only_correct', 'model_only_correct'],
    default='both_wrong',
)
prediction_frame.to_csv(
    RESULT_DIR / 'confirmatory_predictions.csv', index=False
)

linkage_rows = []
for problem_type, frame in prediction_frame.groupby('problem_type'):
    linkage_rows.append({
        'problem_type': problem_type,
        'scene_count': len(frame),
        'model_accuracy': float(frame['model_correct'].mean()),
        'probe_sign_accuracy': float(frame['probe_correct'].mean()),
        'probe_only_correct_rate': float(
            (frame['representation_behavior_case'] == 'probe_only_correct').mean()
        ),
        'weight_heuristic_accuracy': float(
            frame['weight_heuristic_correct'].mean()
        ),
        'model_weight_agreement': float(
            frame['model_follows_weight_heuristic'].mean()
        ),
    })
linkage_by_problem = pd.DataFrame(linkage_rows).sort_values('problem_type')
linkage_by_problem.to_csv(
    RESULT_DIR / 'behavior_linkage_by_problem_type.csv', index=False
)

model_error_mask = ~prediction_frame['model_correct'].to_numpy()
model_error_metrics = None
if model_error_mask.sum() >= 5 and np.std(y_test[model_error_mask]) > 0:
    model_error_metrics = {
        'scene_count': int(model_error_mask.sum()),
        'probe_r2_on_model_errors': float(r2_score(
            y_test[model_error_mask], probe_prediction[model_error_mask]
        )),
        'probe_mae_on_model_errors': float(mean_absolute_error(
            y_test[model_error_mask], probe_prediction[model_error_mask]
        )),
        'probe_sign_accuracy_on_model_errors': float(
            prediction_frame.loc[model_error_mask, 'probe_correct'].mean()
        ),
    }
behavior_linkage_summary = {
    'model_scene_accuracy': float(prediction_frame['model_correct'].mean()),
    'model_scene_balanced_accuracy': float(balanced_accuracy_score(
        prediction_frame['outcome'],
        prediction_frame['model_predicted_outcome'],
    )),
    'probe_sign_accuracy': float(prediction_frame['probe_correct'].mean()),
    'probe_sign_balanced_accuracy': float(balanced_accuracy_score(
        prediction_frame['outcome'],
        prediction_frame['probe_predicted_outcome'],
    )),
    'probe_only_correct_count': int(
        (prediction_frame['representation_behavior_case'] == 'probe_only_correct').sum()
    ),
    'probe_only_correct_rate': float(
        (prediction_frame['representation_behavior_case'] == 'probe_only_correct').mean()
    ),
    'weight_conflict_scene_count': int(
        (~prediction_frame['weight_heuristic_correct']).sum()
    ),
    'model_error_subset': model_error_metrics,
}
display(linkage_by_problem.round(4))
display(pd.Series(behavior_linkage_summary))


In [ ]:
%%cell_progress 15/15 Save figures, verify outputs, and compare models
cell_stage('Finalizing preregistered confirmation status')
r2_interval = bootstrap_intervals[
    bootstrap_intervals['metric'].eq('test_r2')
].iloc[0]
if (
    primary_point_metrics['test_r2'] > 0
    and float(r2_interval['ci95_low']) > 0
    and shuffle_p_value <= 0.05
):
    confirmation_status = 'confirmed'
elif primary_point_metrics['test_r2'] > 0:
    confirmation_status = 'mixed'
else:
    confirmation_status = 'not_confirmed'

confirmatory_summary = {
    'schema_version': 'activation-confirmatory-results-1.0',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'resolved_revision': MODEL_REVISION,
    'fit_splits': ['probe_train', 'probe_dev'],
    'evaluation_split': 'probe_test',
    'test_used_for_selection': False,
    'primary_target': 'net_torque',
    'representation': 'mean of six option-permutation final-prompt states',
    'selected_layer_index': SELECTED_LAYER_INDEX,
    'selected_normalized_depth': SELECTED_NORMALIZED_DEPTH,
    'frozen_ridge_alpha': FROZEN_ALPHA,
    'balance_threshold': BALANCE_THRESHOLD,
    'fit_scene_count': len(y_fit),
    'test_scene_count': len(y_test),
    'point_metrics': primary_point_metrics,
    'bootstrap_intervals': bootstrap_intervals.to_dict(orient='records'),
    'shuffle_null': {
        'replicates': N_SHUFFLES,
        'one_sided_empirical_p_r2': shuffle_p_value,
        'r2_quantile_95': float(shuffle_null['test_r2'].quantile(0.95)),
        'r2_max': float(shuffle_null['test_r2'].max()),
    },
    'confirmation_status': confirmation_status,
    'confirmation_rule': (
        'confirmed iff test R2 > 0, stratified scene-bootstrap R2 lower 95% '
        'bound > 0, and shuffled-label one-sided empirical p <= 0.05'
    ),
    'behavior_linkage': behavior_linkage_summary,
    'interpretation_limit': (
        'Held-out decodability and representation-behavior dissociation do not '
        'establish mediation, causal use, or a torque circuit.'
    ),
}
write_json_atomic(RESULT_DIR / 'confirmatory_summary.json', confirmatory_summary)

cell_stage('Rendering confirmatory figures')
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')

figure, axis = plt.subplots(figsize=(8, 5))
axis.hist(
    shuffle_null['test_r2'], bins=30, color='0.7', edgecolor='white',
    label='shuffled-label null',
)
axis.axvline(
    primary_point_metrics['test_r2'], color='tab:red', linewidth=2,
    label='frozen activation probe',
)
axis.axvline(0, color='black', linestyle='--', linewidth=1)
axis.set_xlabel('probe_test R²')
axis.set_ylabel('Count')
axis.set_title(f'{MODEL_ID}: frozen net-torque probe vs null')
axis.legend()
plt.tight_layout()
plt.savefig(
    RESULT_DIR / '01_test_r2_vs_shuffle_null.png',
    dpi=180, bbox_inches='tight',
)
plt.show()

figure, axis = plt.subplots(figsize=(9, 5))
plot_baselines = shortcut_baselines.sort_values('test_r2')
colors = [
    'tab:red' if name == 'activation_probe' else 'tab:blue'
    for name in plot_baselines['baseline']
]
axis.barh(plot_baselines['baseline'], plot_baselines['test_r2'], color=colors)
axis.axvline(0, color='black', linestyle='--', linewidth=1)
axis.set_xlabel('probe_test R²')
axis.set_title(f'{MODEL_ID}: activation and raw-feature baselines')
plt.tight_layout()
plt.savefig(
    RESULT_DIR / '02_shortcut_baselines.png',
    dpi=180, bbox_inches='tight',
)
plt.show()

figure, axis = plt.subplots(figsize=(11, 5))
positions = np.arange(len(linkage_by_problem))
width = 0.38
axis.bar(
    positions - width / 2, linkage_by_problem['model_accuracy'],
    width=width, label='model answer',
)
axis.bar(
    positions + width / 2, linkage_by_problem['probe_sign_accuracy'],
    width=width, label='probe torque sign',
)
axis.set_xticks(positions)
axis.set_xticklabels(linkage_by_problem['problem_type'], rotation=35, ha='right')
axis.set_ylim(0, 1.05)
axis.set_ylabel('Accuracy')
axis.set_title(f'{MODEL_ID}: representation–behavior linkage on probe_test')
axis.legend()
plt.tight_layout()
plt.savefig(
    RESULT_DIR / '03_representation_behavior_by_problem.png',
    dpi=180, bbox_inches='tight',
)
plt.show()

required_outputs = [
    TEST_CAPTURE_DIR / 'capture_config.json',
    TEST_CAPTURE_DIR / 'protocol_lock.json',
    TEST_CAPTURE_DIR / 'capture_summary.json',
    TEST_CAPTURE_DIR / 'capture_index.jsonl',
    TEST_CAPTURE_DIR / 'probe_test_manifest.jsonl',
    TEST_CAPTURE_DIR / 'probe_test_scene_mean_activations.pt',
    RESULT_DIR / 'confirmatory_summary.json',
    RESULT_DIR / 'confirmatory_predictions.csv',
    RESULT_DIR / 'bootstrap_intervals.csv',
    RESULT_DIR / 'shuffle_null_r2.csv',
    RESULT_DIR / 'shortcut_baselines.csv',
    RESULT_DIR / 'behavior_linkage_by_problem_type.csv',
    RESULT_DIR / '01_test_r2_vs_shuffle_null.png',
    RESULT_DIR / '02_shortcut_baselines.png',
    RESULT_DIR / '03_representation_behavior_by_problem.png',
]
missing = [str(path) for path in required_outputs if not path.is_file()]
assert not missing, 'Missing outputs:\n' + '\n'.join(missing)
run_manifest = {
    'schema_version': 'activation-confirmatory-run-1.0',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'capture_dir': str(TEST_CAPTURE_DIR),
    'result_dir': str(RESULT_DIR),
    'discovery_selection_path': str(DISCOVERY_SELECTION_PATH),
    'discovery_selection_sha256': sha256_file(DISCOVERY_SELECTION_PATH),
    'protocol_lock_path': str(PROTOCOL_LOCK_PATH),
    'protocol_lock_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'probe_test_opened': True,
    'selection_frozen_before_probe_test': True,
    'confirmation_status': confirmation_status,
    'required_outputs': [str(path) for path in required_outputs],
}
write_json_atomic(RESULT_DIR / 'run_manifest.json', run_manifest)

cell_stage('Checking whether both confirmatory model runs are available')
comparison_specs = {
    'Qwen3.5-9B': (
        PROJECT_ROOT / 'results' / 'Qwen__Qwen3.5-9B'
        / 'activation_confirmatory_v1' / 'confirmatory_results'
        / 'confirmatory_summary.json'
    ),
    'Gemma4-12B': (
        PROJECT_ROOT / 'results' / 'google__gemma-4-12B-it'
        / 'activation_confirmatory_v1' / 'confirmatory_results'
        / 'confirmatory_summary.json'
    ),
}
if all(path.is_file() for path in comparison_specs.values()):
    comparison_rows = []
    for name, path in comparison_specs.items():
        summary = json.loads(path.read_text(encoding='utf-8'))
        intervals = {
            row['metric']: row for row in summary['bootstrap_intervals']
        }
        comparison_rows.append({
            'model_name': name,
            'confirmation_status': summary['confirmation_status'],
            'test_r2': summary['point_metrics']['test_r2'],
            'test_r2_ci_low': intervals['test_r2']['ci95_low'],
            'test_r2_ci_high': intervals['test_r2']['ci95_high'],
            'probe_outcome_balanced_accuracy': (
                summary['point_metrics']['test_outcome_balanced_accuracy']
            ),
            'model_outcome_balanced_accuracy': (
                summary['behavior_linkage']['model_scene_balanced_accuracy']
            ),
            'probe_only_correct_rate': (
                summary['behavior_linkage']['probe_only_correct_rate']
            ),
            'shuffle_p_r2': (
                summary['shuffle_null']['one_sided_empirical_p_r2']
            ),
        })
    comparison = pd.DataFrame(comparison_rows)
    comparison_dir = (
        PROJECT_ROOT / 'results' / 'activation_confirmatory_comparison_v1'
    )
    comparison_dir.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(
        comparison_dir / 'confirmatory_model_comparison.csv', index=False
    )
    display(comparison.round(4))
    print('Cross-model comparison:', comparison_dir)
else:
    missing_models = [
        name for name, path in comparison_specs.items() if not path.is_file()
    ]
    print('Cross-model comparison deferred. Still missing:', missing_models)
    print('Run this notebook for the other model in a fresh runtime.')

print('Confirmatory run complete:', MODEL_ID)
print('Status:', confirmation_status)
print('Next gate: inspect held-out results before building causal activation patching.')
